In [2]:
import cv2
import mediapipe as mp
import time

mp_pose = mp.solutions.pose
mp_draw = mp.solutions.drawing_utils
pose = mp_pose.Pose()


prev_y = None
fall_threshold = 10
fall_detected = False



cap = cv2.VideoCapture('../data/videos/fall.mp4')

output_path = 'fall_detection.avi'
out = None



while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = pose.process(img_rgb)

    if results.pose_landmarks:
        landmarks = results.pose_landmarks.landmark
        mp_draw.draw_landmarks(frame, results.pose_landmarks, mp_pose.POSE_CONNECTIONS)


        
        left_hip = landmarks[mp_pose.PoseLandmark.LEFT_HIP]
        y = int(left_hip.y * frame.shape[0])

        
        #to define falls if dy is less threshold;
        if prev_y is not None:
            dy = y - prev_y
            if dy > fall_threshold:
                fall_detected = True
        # 'Sitting' 상태 감지 로직
        # 'fall.mp4' 영상의 초반부에 앉아 있는 상태를 기준으로
        # 'fall_detected'가 False이고, 엉덩이의 Y 좌표가 일정 범위 내에 있을 때 'Sitting'으로 간주
        if not fall_detected and y < 500: # y<450은 앉아있는 자세의 y좌표를 대략적으로 파악하여 설정한 값입니다.
            cv2.putText(frame, 'Sitting', (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 
                        1, (0, 255, 0), 3)
        prev_y = y

        


    
    if fall_detected:
        cv2.putText(frame, 'Fall Detected', (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 
                    1, (0,0,255), 3)

    if out is None:
        height, width, _ = frame.shape
        out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'XVID'), 10, (width, height))

    out.write(frame)
    cv2.imshow('frame', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
if out:
    out.release()
cv2.destroyAllWindows()